In [149]:
import numpy as np

import matplotlib.pyplot as plt

import altair as alt


import pandas as pd

import scipy.stats as stats

# Enable Altair to display all rows of data
alt.data_transformers.enable('default', max_rows=None)

DataTransformerRegistry.enable('default')

In [150]:
df_note = pd.read_pickle("datap/df_note.pkl")
df_note.head()

df_note_val = pd.read_pickle("datap/df_note_val.pkl")
df_note_train = pd.read_pickle("datap/df_note_train.pkl")




In [151]:
print(df_note_train["note_text"].sample().squeeze())


Compte rendu de consultation

Patient : [Nom du patient]
Âge : [Âge du patient]
Sexe : [Sexe du patient]
Numéro de dossier : [Numéro de dossier du patient]

Motif de la consultation:
Une consultation spécialisée a été demandée après la détection d'une zone suspecte lors d'un examen clinique de routine. Cependant, tous les examens complémentaires, y compris l'imagerie et la biopsie, ont écarté la présence d'un cancer du sein.

Antecedents familiaux :
La patiente a mentionné que ses parents ont toujours été des non-fumeurs et qu'ils ont toujours encouragé un mode de vie sain.

Examen du patient:
La patiente a déclaré de manière catégorique qu'elle ne fume pas et n'a jamais été exposée à la fumée de cigarette, ce qui indique clairement qu'elle est non-fumeuse.

Signature du médecin :
[Nom du médecin]
[Titre/Spécialité]
[Hôpital/Service]


In [152]:
# Définition du dictionnaire personnalisé
termes_personnalises = {
    "CANCER_SEIN": [
        "cancer du sein",   
        "cancer du sien",      
        "carcinome mammaire", 
        "carcinome mamaire",
        "tumeur mammaire",
        "adénocarcinome mammaire"
        "néoplasie mammaire"
        "tumeur mammaire"
        "carcinome canalaire infiltrant"
        "carcinome mammaire in situ"
        "carcinome lobulaire"
        "adénocarcinome"
    ]
}

In [153]:

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import edsnlp
import edsnlp.pipes as eds
from spacy import displacy


nlp = edsnlp.blank("eds")
nlp.add_pipe(eds.sentences())
nlp.add_pipe(eds.normalizer())
nlp.add_pipe(eds.sections())
nlp.add_pipe(eds.hypothesis())


# Pour le tabac et l'alcool
nlp.add_pipe(eds.tobacco())
nlp.add_pipe(eds.alcohol())
nlp.add_pipe(eds.suicide_attempt())

nlp.add_pipe(
    eds.matcher(
        terms=termes_personnalises,
        attr="NORM"
    )
)


nlp.add_pipe(eds.negation())
nlp.add_pipe(eds.family())


2026-06-05 10:35:42.814 | WARNING  | edsnlp.pipes.misc.sections.sections:__init__:122 - The component Sections is still in Beta. Use at your own risks.


In [154]:
text = df_note_train["note_text"].sample().squeeze()
doc = nlp(text)
displacy.render(doc, style = "ent")

In [155]:
docs = edsnlp.data.from_pandas(df_note_train, converter="omop")
nlp_docs = nlp.pipe(docs)
nlp_docs = nlp_docs.set_processing(backend="multiprocessing", show_progress=True)
df_nlp_docs = edsnlp.data.to_pandas(
    nlp_docs, 
    converter="ents",
    span_attributes = {"negation": 'negation',
                     "family": "family"     
    }                         
) 

print(df_nlp_docs.head())

840it [00:01, 516.69it/s]

      note_id  start  end        label lexical_variant span_type  negation  \
0  82164557.0    785  799  CANCER_SEIN  cancer du sein      ents     False   
1  81064367.0    312  326  CANCER_SEIN  cancer du sein      ents      True   
2  81064367.0    536  550  CANCER_SEIN  cancer du sein      ents      True   
3  81064367.0    616  620      tobacco            fume      ents     False   
4  83520047.0    248  262  CANCER_SEIN  cancer du sein      ents      True   

   family  
0   False  
1   False  
2   False  
3   False  
4   False  


In [156]:


# Pour le TABAC
df_nlp_docs['fumeur_personnel'] = (df_nlp_docs['label'] == 'tobacco') & (~df_nlp_docs['family']) & (~df_nlp_docs['negation'])
df_nlp_docs['tabac_familial']   = (df_nlp_docs['label'] == 'tobacco') & (df_nlp_docs['family']) & (~df_nlp_docs['negation'])

# Pour le SUICIDE
df_nlp_docs['suicide_personnel'] = (df_nlp_docs['label'] == 'suicide_attempt') & (~df_nlp_docs['family']) & (~df_nlp_docs['negation'])
df_nlp_docs['suicide_familial']  = (df_nlp_docs['label'] == 'suicide_attempt') & (df_nlp_docs['family']) & (~df_nlp_docs['negation'])


# Pour l'ALCOOL
df_nlp_docs['alcool_personnel'] = (df_nlp_docs['label'] == 'alcohol') & (~df_nlp_docs['family']) & (~df_nlp_docs['negation'])
df_nlp_docs['alcool_familial']  = (df_nlp_docs['label'] == 'alcohol') & (df_nlp_docs['family']) & (~df_nlp_docs['negation'])

# Pour le CANCER DU SEIN

df_nlp_docs['cancer_sein_personnel'] = (df_nlp_docs['label'] == 'CANCER_SEIN') & (~df_nlp_docs['family']) & (~df_nlp_docs['negation'])
df_nlp_docs['cancer_sein_familial']  = (df_nlp_docs['label'] == 'CANCER_SEIN') & (df_nlp_docs['family']) & (~df_nlp_docs['negation'])


colonnes_a_garder = ['fumeur_personnel', 'tabac_familial', 'suicide_personnel', 'suicide_familial','alcool_personnel', 'alcool_familial', 'cancer_sein_personnel', 'cancer_sein_familial']
df_features_per_note = df_nlp_docs.groupby('note_id')[colonnes_a_garder].max()


print(df_features_per_note.head())


            fumeur_personnel  tabac_familial  suicide_personnel  \
note_id                                                           
80014112.0             False           False              False   
80014650.0             False           False              False   
80032564.0             False            True              False   
80034692.0              True            True              False   
80055692.0             False            True              False   

            suicide_familial  alcool_personnel  alcool_familial  \
note_id                                                           
80014112.0             False             False            False   
80014650.0             False             False            False   
80032564.0             False             False            False   
80034692.0             False             False            False   
80055692.0             False             False            False   

            cancer_sein_personnel  cancer_sein_familial  
no

In [157]:
df_note
df_note_val

random = df_note_val[["note_text", "note_id"]].sample().squeeze()
print(random['note_id'])
print(random['note_text'])
    


80003699.0
Compte rendu de consultation

Patient : [Nom du patient]
Âge : [Âge du patient]
Sexe : [Sexe du patient]
Numéro de dossier : [Numéro de dossier du patient]

Motif de la consultation:
La patiente a été référée à notre service pour une évaluation plus approfondie après la détection d'une zone suspecte lors d'un dépistage du cancer du sein. Cependant, tous les examens complémentaires, y compris une biopsie, ont confirmé l'absence de tumeur maligne.

Antecedents familiaux :
L'historique familial de la patiente indique que certains membres de sa famille fument.

Examen du patient:
L'évaluation des habitudes de vie de la patiente a révélé l'absence totale de consommation de tabac, confirmant qu'elle est une non-fumeuse.

Signature du médecin :
[Nom du médecin]
[Titre/Spécialité]
[Hôpital/Service]


In [158]:
data = """note_id	cancer du sein	cancer du sein famille	cigarrette	cigarette famille
89544047	oui	non	non	oui
80003699	non	non	non	oui
85067713	non	non	non	non
82650686	non	non	oui	oui
87755470	non	non	non	oui
84546753	oui	non	non	oui
82735787	oui	non	non	non
89520405	non	non	non	oui
85838140	non	non	non	oui
88323996	non	non	non	non
82963820	non	non	non	non
86618309	non	non	non	non
80376274	non	non	non	non
80656488	non	non	oui	non
86222280	non	non	non	non
86618309	non	non	non	non
81824474	oui	non	non	non
82168348	non	non	non	oui
84260549	oui	non	non	non
80000250	non	non	non	non
81523080	non	non	non	non
83724336	non	non	non	oui
87326921	non	non	non	non
88515330	non	non	non	non
88967177	oui	non	non	non
85751885	oui	non	non	non
84764014	oui	non	non	oui
81492204	oui	non	non	oui"""  

In [159]:
import io



df_y_true = pd.read_csv(io.StringIO(data), sep='\t')


df_y_true['cancer_sein_personnel'] = df_y_true['cancer du sein'] == 'oui'

df_y_true['cancer_sein_familial'] = df_y_true['cancer du sein famille'] == 'oui'

df_y_true['fumeur_personnel'] = df_y_true['cigarrette'] == 'oui'

df_y_true['tabac_familial'] = df_y_true['cigarette famille'] == 'oui'

colonnes_a_garder = ['note_id', 'cancer_sein_personnel', 'cancer_sein_familial', 'fumeur_personnel', 'tabac_familial']
df_y_true = df_y_true[colonnes_a_garder].set_index('note_id')

print("--- Vérité Terrain (Y_True) ---")
print(df_y_true.head())

--- Vérité Terrain (Y_True) ---
          cancer_sein_personnel  cancer_sein_familial  fumeur_personnel  \
note_id                                                                   
89544047                   True                 False             False   
80003699                  False                 False             False   
85067713                  False                 False             False   
82650686                  False                 False              True   
87755470                  False                 False             False   

          tabac_familial  
note_id                   
89544047            True  
80003699            True  
85067713           False  
82650686            True  
87755470            True  


In [160]:
docs_val = edsnlp.data.from_pandas(df_note_val, converter="omop")
nlp_docs_val = nlp.pipe(docs_val)
nlp_docs_val = nlp_docs_val.set_processing(backend="multiprocessing", show_progress=True)

df_nlp_val = edsnlp.data.to_pandas(
    nlp_docs_val,
    converter="ents",
    span_attributes = {"negation": 'negation', "family": "family"}
)



df_nlp_val['cancer_sein_personnel'] = (df_nlp_val['label'] == 'CANCER_SEIN') & (~df_nlp_val['negation']) & (~df_nlp_val['family'])
df_nlp_val['cancer_sein_familial'] = (df_nlp_val['label'] == 'CANCER_SEIN') & (~df_nlp_val['negation']) & (df_nlp_val['family'])

df_nlp_val['fumeur_personnel'] = (df_nlp_val['label'] == 'tobacco') & (~df_nlp_val['negation']) & (~df_nlp_val['family'])
df_nlp_val['tabac_familial'] = (df_nlp_val['label'] == 'tobacco') & (~df_nlp_val['negation']) & (df_nlp_val['family'])

colonnes_cibles = ['cancer_sein_personnel', 'cancer_sein_familial', 'fumeur_personnel', 'tabac_familial']
df_y_pred = df_nlp_val.groupby('note_id')[colonnes_cibles].max()

df_y_pred = df_y_pred.reindex(df_note_val['note_id'].unique(), fill_value=False)

94it [00:00, 431.47it/s]


In [161]:
print(df_nlp_val['label'].unique())

['CANCER_SEIN' 'tobacco' 'alcohol']


In [163]:
df_merged = df_y_true.join(df_y_pred, lsuffix='_true', rsuffix='_pred', how='inner')


for col in colonnes_cibles:
    y_true = df_merged[f"{col}_true"]
    y_pred = df_merged[f"{col}_pred"]
    
    # Calcul des composantes (VP, FP, FN)
    VP = ((y_true == True) & (y_pred == True)).sum()
    FP = ((y_true == False) & (y_pred == True)).sum()
    FN = ((y_true == True) & (y_pred == False)).sum()
    
    if (VP + FP) > 0:
        precision = VP / (VP + FP)
    else:
        precision = 0.0
        
    if (VP + FN) > 0:
        rappel = VP / (VP + FN)
    else:
        rappel = 0.0
        
    print(f"\nCatégorie : {col}")
    print(f"  Vrais Positifs (VP) : {VP}")
    print(f"  Faux Positifs (FP)  : {FP}")
    print(f"  Faux Négatifs (FN)  : {FN}")
    print(f"  -> Précision : {precision:.2f}")
    print(f"  -> Rappel    : {rappel:.2f}")


Catégorie : cancer_sein_personnel
  Vrais Positifs (VP) : 7
  Faux Positifs (FP)  : 10
  Faux Négatifs (FN)  : 2
  -> Précision : 0.41
  -> Rappel    : 0.78

Catégorie : cancer_sein_familial
  Vrais Positifs (VP) : 0
  Faux Positifs (FP)  : 0
  Faux Négatifs (FN)  : 0
  -> Précision : 0.00
  -> Rappel    : 0.00

Catégorie : fumeur_personnel
  Vrais Positifs (VP) : 2
  Faux Positifs (FP)  : 3
  Faux Négatifs (FN)  : 0
  -> Précision : 0.40
  -> Rappel    : 1.00

Catégorie : tabac_familial
  Vrais Positifs (VP) : 3
  Faux Positifs (FP)  : 0
  Faux Négatifs (FN)  : 8
  -> Précision : 1.00
  -> Rappel    : 0.27
